In [1]:
!pip install transformers datasets sentencepiece accelerate -q

In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import load_dataset

In [3]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [4]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
small_train = dataset["train"].select(range(5000))
small_val = dataset["validation"].select(range(500))

print(f"Train size: {len(small_train)}")
print(f"Validation size: {len(small_val)}")

Train size: 5000
Validation size: 500


In [ ]:
## preprocess /tokenize data
MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(
        inputs, 
        max_length = MAX_INPUT,
        truncation = True,
        padding = "max_length",
        
    )
   # with tokenizer.as_target_tokenizer(): // problem kr rha h no attribut aa rh ah
    label_encodings = tokenizer(
    examples["highlights"],
    max_length = MAX_TARGET,
    truncation = True,
    padding = "max_length",
    
)

    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in label_encodings["input_ids"]
    ]
    
    return model_inputs

In [7]:
##Apply preprocessing
# 
tokenized_train = small_train.map(
    preprocess,
    batched = True,
    remove_columns = small_val.column_names
) 
tokenized_val = small_val.map(
    preprocess,
    batched=True,
    remove_columns= small_val.column_names
)

print("tokenization done!")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

tokenization done!


In [8]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./t2-notes-summerizer",
    num_train_epochs= 3,
    per_device_train_batch_size=4,
     per_device_eval_batch_size=4,
     warmup_steps=200,
     weight_decay= 0.01,
     logging_steps=50,
     eval_strategy="epoch",
     save_strategy="epoch",
     load_best_model_at_end=True,
     fp16=True,
     predict_with_generate=True,
     report_to="none"
)

In [9]:
# Training setup
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding= True)

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val,
    processing_class = tokenizer,
    data_collator = data_collator,
)

In [ ]:
# lebels checks
print(type(tokenized_train[0]["labels"]))
print(tokenized_train[0]["labels"][:10])

<class 'list'>
[8929, 16023, 2213, 4173, 6324, 12591, 15, 2347, 3996, 1755]


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.203200,2.173884
2,2.044800,2.173616
3,2.112400,2.176044


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3750, training_loss=2.117670751953125, metrics={'train_runtime': 475.8957, 'train_samples_per_second': 31.52, 'train_steps_per_second': 7.88, 'total_flos': 2030127022080000.0, 'train_loss': 2.117670751953125, 'epoch': 3.0})

In [12]:
# save the model
model.save_pretrained("./t5-note-summarizer")
tokenizer.save_pretrained("./t5-note-summarizer")
print("model saved!")

model saved!


In [15]:
# test
def summarize(text):
    input_text = "summarize: " + text
    inputs = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length = 512,
        truncation = True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        num_beams = 4,
        early_stopping = True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

my_notes = """
Deep Learning is a specialized area of Machine Learning that focuses on algorithms 
inspired by the structure and function of the human brain, known as Artificial Neural Networks. 
The main idea behind deep learning is to enable computers to learn patterns from large amounts 
of data and make intelligent decisions without being explicitly programmed for every task.
Traditional machine learning methods often require manual feature extraction, where experts 
define the important characteristics of the data. In contrast, deep learning models automatically 
learn useful features from raw data through multiple layers of neural networks. Because these 
models contain many hidden layers, the learning process is called deep learning.
Deep learning has become extremely important in modern artificial intelligence because of its 
ability to process large volumes of data and solve complex problems such as image recognition, 
speech recognition, language translation, and autonomous driving.
The foundation of deep learning is the Artificial Neural Network. Neural networks are 
computational models inspired by biological neural networks found in the human brain. 
A neural network consists of several layers of interconnected nodes called neurons which 
are divided into input layer, hidden layers, and output layer.
"""

print(summarize(my_notes))

Deep learning is a specialized area of Machine Learning that focuses on algorithms inspired by the structure and function of the human brain, known as Artificial Neural Networks. Deep learning models automatically learn useful features from raw data through multiple layers of neural networks. Deep learning is extremely important in modern artificial intelligence because of its ability to process large volumes of data.


In [16]:
import re

def summary_to_points(text):
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    points = [f"• {s.strip()}" for s in sentences if s.strip()]
    return "\n".join(points)


result = summarize(my_notes)
points = summary_to_points(result)
print(points)

• Deep learning is a specialized area of Machine Learning that focuses on algorithms inspired by the structure and function of the human brain, known as Artificial Neural Networks.
• Deep learning models automatically learn useful features from raw data through multiple layers of neural networks.
• Deep learning is extremely important in modern artificial intelligence because of its ability to process large volumes of data.
